In [1]:
import torch
from transformers import AutoConfig, AutoImageProcessor, AutoModelForVision2Seq, AutoProcessor
import time
import numpy as np
import cv2
import textwrap
from PIL import Image, ImageDraw, ImageFont
import enum
import json
import os 
import asyncio
from uuid import uuid4

/home/zhekai/miniforge3/envs/openvla-vllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-24 13:19:26.683291: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-24 13:19:26.714119: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-24 13:19:26.714143: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-2

In [2]:
class CotTag(enum.Enum):
    TASK = "TASK:"
    PLAN = "PLAN:"
    VISIBLE_OBJECTS = "VISIBLE OBJECTS:"
    SUBTASK_REASONING = "SUBTASK REASONING:"
    SUBTASK = "SUBTASK:"
    MOVE_REASONING = "MOVE REASONING:"
    MOVE = "MOVE:"
    GRIPPER_POSITION = "GRIPPER POSITION:"
    ACTION = "ACTION:"

def get_cot_tags_list():
    return [
        CotTag.TASK.value,
        CotTag.PLAN.value,
        CotTag.VISIBLE_OBJECTS.value,
        CotTag.SUBTASK_REASONING.value,
        CotTag.SUBTASK.value,
        CotTag.MOVE_REASONING.value,
        CotTag.MOVE.value,
        CotTag.GRIPPER_POSITION.value,
        CotTag.ACTION.value,
    ]



In [3]:
device = "cuda:0"
# Load Processor & VLA
path_to_converted_ckpt = "Embodied-CoT/ecot-openvla-7b-oxe"
# path_to_converted_ckpt = "/home/zhekai/code/embodied-CoT/outputs/ecot-openvla-7b-oxe+libero_object_no_noops+b1+lr-0.0005+lora-r32+dropout-0.0"
processor = AutoProcessor.from_pretrained(path_to_converted_ckpt, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    path_to_converted_ckpt,
    torch_dtype=torch.bfloat16,
    # low_cpu_mem_usage=True,
    trust_remote_code=True,
).to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Expected `transformers==4.40.1` and `tokenizers==0.19.1` but got `transformers==4.49.0` and `tokenizers==0.21.1`; there might be inference-time regressions due to dependency changes. If in doubt, pleaseuse the above versions.
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.74it/s]


In [4]:
# vla.language_model.save_pretrained("logs/llama-bridge")
from vllm import LLM, SamplingParams, AsyncEngineArgs, AsyncLLMEngine
from vllm.inputs import TokensPrompt
vla.input_embds = vla.language_model.get_input_embeddings()

# load language model with VLLM
if hasattr(vla, "language_model"):
    del vla.language_model


INFO 03-24 13:19:41 __init__.py:186] Automatically detected platform cuda.


2025-03-24 13:19:41,685	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [5]:
sampling_params = SamplingParams(temperature=0, max_tokens=60, stop_token_ids=[29901])
async_engine = AsyncLLMEngine.from_engine_args(
        AsyncEngineArgs(
            model="../logs/llama-bridge",
            gpu_memory_utilization=0.7,
            preemption_mode="swap",
            swap_space=10,
        )
)


INFO 03-24 13:19:47 config.py:542] This model supports multiple tasks: {'embed', 'generate', 'reward', 'score', 'classify'}. Defaulting to 'generate'.
INFO 03-24 13:19:47 llm_engine.py:234] Initializing a V0 LLM engine (v0.1.dev4429+ga257914) with config: model='../logs/llama-bridge', speculative_config=None, tokenizer='../logs/llama-bridge', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../logs/llama-bridge, num_sch

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:00,  2.42it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.91it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  1.78it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  1.85it/s]



INFO 03-24 13:19:50 model_runner.py:1119] Loading model weights took 12.5528 GB
INFO 03-24 13:19:51 worker.py:267] Memory profiling takes 0.85 seconds
INFO 03-24 13:19:51 worker.py:267] the current vLLM instance can use total_gpu_memory (23.64GiB) x gpu_memory_utilization (0.70) = 16.55GiB
INFO 03-24 13:19:51 worker.py:267] model weights take 12.55GiB; non_torch_memory takes 0.08GiB; PyTorch activation peak memory takes 0.44GiB; the rest of the memory reserved for KV Cache is 3.48GiB.
INFO 03-24 13:19:52 executor_base.py:110] # CUDA blocks: 445, # CPU blocks: 1280
INFO 03-24 13:19:52 executor_base.py:115] Maximum concurrency for 2048 tokens per request: 3.48x
INFO 03-24 13:19:56 model_runner.py:1438] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:17<00:00,  2.04it/s]

INFO 03-24 13:20:13 model_runner.py:1566] Graph capturing finished in 17 secs, took 0.24 GiB
INFO 03-24 13:20:13 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 22.53 seconds


In [6]:
SYSTEM_PROMPT = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)
t = CotTag.TASK.value
def get_openvla_prompt(instruction: str, task) -> str:
    return f"{SYSTEM_PROMPT} USER: What action should the robot take to {instruction.lower()}? ASSISTANT: {task}"
INSTRUCTION = "place the watermelon on the towel"
prompt = get_openvla_prompt(INSTRUCTION, t)
image = Image.open("../test.png")
print(prompt.replace(". ", ".\n"))
# print("Image size:", image.size)
dataset_statistics_path = os.path.join(path_to_converted_ckpt, "dataset_statistics.json")
if os.path.isfile(dataset_statistics_path):
    with open(dataset_statistics_path, "r") as f:
        norm_stats = json.load(f)
    vla.norm_stats = norm_stats

A chat between a curious user and an artificial intelligence assistant.
The assistant gives helpful, detailed, and polite answers to the user's questions.
USER: What action should the robot take to place the watermelon on the towel? ASSISTANT: TASK:


In [7]:
async def engine_inference(
    model,
    engine,
    input_ids = None,
    pixel_values = None,
    sampling_params = None,
):
    # Visual Feature Extraction (shared across batched lanuaged inputs)
    patch_features = model.vision_backbone(pixel_values)
    projected_patch_embeddings = model.projector(patch_features)
    embds = model.input_embds
    input_embeddings = [embds(ids) for ids in input_ids]
    
    # Build Multimodal Embeddings & Attention Mask =>> Prismatic defaults to inserting after <BOS> token (1:)
    multimodal_embeddings = [torch.cat([inemb[:, :1, :], projected_patch_embeddings, inemb[:, 1:, :]], dim=1).squeeze(0) for inemb in input_embeddings]#[0] 
    prompt = [[32000] * emb.shape[-2] for emb in multimodal_embeddings]
    inputs = [{"prompt_token_ids": p, "multi_modal_data": {"image":m}} for p, m in zip(prompt, multimodal_embeddings)]
    tasks = [asyncio.create_task(run_query(TokensPrompt(**q), engine, sampling_params)) for q in inputs]
    results = []
    for task in asyncio.as_completed(tasks):
        result = await task
        results.append(result)
    return results

async def run_query(query, engine, params):
    request_id = uuid4()
    outputs = engine.generate(query, params, request_id)
    async for output in outputs:
        final_output = output
    responses = []
    for output in final_output.outputs:
        responses.append(output.text)
    return responses



different prompt tests (not directly supported)

prepare the inputs 

In [8]:
async_prompts = "A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: What action should the robot take to place the watermelon on the towel? ASSISTANT: TASK: The task is to place the watermelon on the towel. The first step is to move the robotic arm towards the towel. PLAN: 1. Move to the right and forward. 2. Move down and grip the towel. 3. Move backward and up. 4. Move to the left. VISIBLE OBJECTS: the robot task [100, 1, 153, 105], the towel [160, 99, 220, 164], the towel [160, 99, 221, 165], table [20, 39, 239, 249], the robot task [100, 1, 154, 106] SUBTASK REASONING: The towel is to the right and slightly forward from the current robotic arm position. The robotic arm needs to move forward and up to reach the towel and grip it. SUBTASK: Move forward and up. MOVE REASONING: The robotic arm needs to move forward and up to reach the towel and grip it. MOVE: Move forward up. GRIPPER POSITION: [121, 91, 130, 87, 142, 87, 153, 88, 169, 95] ACTION: 塔瀬ܝĦ越ਿŸ"
# break async_prompts with CotTag keep value before the tag

prompts = []
for t in CotTag:
    # if t == CotTag.PLAN:
    #     break
    prompts.append(async_prompts.split(t.value)[0] + t.value)
    # print(prompts[-1]) 
from transformers.utils import TensorType
prompts_reason = prompts[:-1]
prompts_action = prompts[-1]

inputs_reason = [processor.tokenizer(p, return_tensors=TensorType.PYTORCH)['input_ids'].to(device) for p in prompts_reason]
inputs_action = [processor.tokenizer(prompts_action, return_tensors=TensorType.PYTORCH)['input_ids'].to(device)]
pixel_values = processor.image_processor(image, return_tensors=TensorType.PYTORCH)["pixel_values"].to(device, dtype=torch.bfloat16)

In [9]:
print(processor.tokenizer('VISIBLE OBJECTS:'))
print(processor.tokenizer.decode([29901]))

{'input_ids': [1, 478, 3235, 8979, 1307, 438, 29933, 17637, 29903, 29901], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
:


In [15]:
start = time.perf_counter()
results = await engine_inference(vla, async_engine, inputs_action, pixel_values, sampling_params)
end = time.perf_counter()
print(f"Time: {end - start}")
print(results)

INFO 03-24 13:25:09 async_llm_engine.py:211] Added request 1eb003d4-55b7-4b08-9ef6-6c5adda11774.
Time: 0.2583774150116369
[['్止群ữḳ𝓝Ÿ']]


In [21]:
def get_outputs(model, engine, inputs, pixel_values, sampling_params, loop):    
    start = time.perf_counter()
    result = asyncio.run_coroutine_threadsafe(engine_inference(model, engine, inputs, pixel_values, sampling_params), loop)
    print("Inference time:", time.perf_counter() - start)
    print(result)

In [22]:
import threading
sampling_params = SamplingParams(temperature=0, max_tokens=60, stop_token_ids=[29901])
loop = asyncio.get_event_loop()
t1 = threading.Thread(target=get_outputs, args=(vla, async_engine, inputs_action, pixel_values, sampling_params, loop), daemon=True)
t2 = threading.Thread(target=get_outputs, args=(vla, async_engine, inputs_reason, pixel_values, sampling_params, loop), daemon=True)
t1.start()
t2.start()
t1.join()
t2.join()

Inference time: 0.00041412701830267906
<Future at 0x72bdcab67390 state=pending>
Inference time: 9.840703569352627e-05
<Future at 0x72bdd072be10 state=pending>


INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 394a21a6-24bc-43da-8b65-55d243061398.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 6686544f-3358-46df-95e9-7110bed824cc.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 2ac6f14a-92c9-4e5f-b9c4-9a113e0fa0d0.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 473c08da-b39a-447e-a18f-99ba31b8db53.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 92f49843-10ec-43ad-99cd-41ce2464a3a8.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 48515b6a-e328-4d23-94a2-eea1e7cbfe62.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request c7e75f1e-d44e-4d49-9386-1532e3a8a7a6.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 67df2dd7-95f8-494f-be35-23155464832b.
INFO 03-23 19:19:31 async_llm_engine.py:211] Added request 29e11a04-8176-4ca9-8bae-27527c759476.
